# RealSelf Crawler — Google Colab

Crawls RealSelf before/after reviews using Playwright with stealth settings.
Run on **Google Colab** (fresh IP bypasses RealSelf bot detection).

**Output:** `realself_pairs.db` — download and merge with your local `staging.db`
using `scripts/merge_queue.py` before uploading to Kaggle.

**Runtime:** ~2–4 hours depending on how many pages load before rate limiting.

In [ ]:
# Cell 1: Install dependencies
!pip install -q playwright beautifulsoup4 lxml loguru
!playwright install chromium --with-deps -q
print("Done")

In [ ]:
# Cell 2: Setup DB and helpers

import sqlite3, json, time, random
from datetime import datetime
from pathlib import Path
from bs4 import BeautifulSoup
from loguru import logger

DB_PATH    = Path('/content/realself_pairs.db')
RATE_LIMIT = 7.0   # seconds between page requests

conn = sqlite3.connect(DB_PATH)
conn.execute("""
    CREATE TABLE IF NOT EXISTS staging_queue (
        id           INTEGER PRIMARY KEY AUTOINCREMENT,
        before_url   TEXT NOT NULL,
        after_url    TEXT NOT NULL,
        source_url   TEXT NOT NULL,
        source_name  TEXT NOT NULL DEFAULT 'realself',
        language     TEXT NOT NULL DEFAULT 'en',
        consent_tier INTEGER NOT NULL DEFAULT 1,
        metadata     TEXT,
        status       TEXT NOT NULL DEFAULT 'pending',
        failure_reason TEXT,
        created_at   TEXT NOT NULL,
        processed_at TEXT,
        UNIQUE(before_url, after_url)
    )
""")
conn.commit()

SLUG_TO_TREATMENT = {
    'botox':           'botox',
    'dysport':         'botox',
    'xeomin':          'botox',
    'juvederm':        'dermal_filler',
    'restylane':       'dermal_filler',
    'sculptra':        'dermal_filler',
    'lip-augmentation':'lip_filler',
    'lip-filler':      'lip_filler',
    'cheek-augmentation':'dermal_filler',
    'under-eye-filler':'under_eye_filler',
    'kybella':         'kybella',
    'chin-filler':     'jawline_filler',
    'rhinoplasty':     'rhinoplasty',
    'facelift':        'facelift',
    'blepharoplasty':  'blepharoplasty',
    'brow-lift':       'facelift',
    'neck-lift':       'facelift',
    'thread-lift':     'thread_lift',
    'chemical-peel':   'chemical_peel',
    'laser-skin-resurfacing': 'laser_resurfacing',
    'microneedling':   'microneedling',
}

SEED_URLS = [
    # Injectables
    'https://www.realself.com/reviews/botox',
    'https://www.realself.com/reviews/dysport',
    'https://www.realself.com/reviews/xeomin',
    'https://www.realself.com/reviews/juvederm',
    'https://www.realself.com/reviews/restylane',
    'https://www.realself.com/reviews/sculptra',
    'https://www.realself.com/reviews/lip-augmentation',
    'https://www.realself.com/reviews/lip-filler',
    'https://www.realself.com/reviews/cheek-augmentation',
    'https://www.realself.com/reviews/under-eye-filler',
    'https://www.realself.com/reviews/kybella',
    'https://www.realself.com/reviews/chin-filler',
    # Surgical
    'https://www.realself.com/reviews/rhinoplasty',
    'https://www.realself.com/reviews/facelift',
    'https://www.realself.com/reviews/blepharoplasty',
    'https://www.realself.com/reviews/brow-lift',
    'https://www.realself.com/reviews/neck-lift',
    # Skin
    'https://www.realself.com/reviews/thread-lift',
    'https://www.realself.com/reviews/chemical-peel',
    'https://www.realself.com/reviews/laser-skin-resurfacing',
    'https://www.realself.com/reviews/microneedling',
]

def extract_pairs(html: str, page_url: str) -> list[dict]:
    soup = BeautifulSoup(html, 'lxml')
    slug = page_url.rstrip('/').split('?')[0].split('/')[-1]
    treatment = SLUG_TO_TREATMENT.get(slug)
    pairs = []
    for card in soup.select('[data-before-photo][data-after-photo]'):
        before_url = card.get('data-before-photo', '').strip()
        after_url  = card.get('data-after-photo', '').strip()
        if not before_url or not after_url:
            continue
        meta = {}
        if treatment:
            meta['treatment_category'] = treatment
        el = card.select_one('[data-treatment-name]')
        if el:
            meta['treatment_name'] = el.get('data-treatment-name', '')
        pairs.append({'before_url': before_url, 'after_url': after_url,
                      'source_url': page_url, 'metadata': json.dumps(meta)})
    return pairs

def save_pairs(pairs: list[dict]) -> tuple[int, int]:
    ins = skip = 0
    for p in pairs:
        try:
            conn.execute(
                'INSERT INTO staging_queue '
                '(before_url,after_url,source_url,metadata,created_at) '
                'VALUES (?,?,?,?,?)',
                (p['before_url'], p['after_url'], p['source_url'],
                 p['metadata'], datetime.utcnow().isoformat())
            )
            ins += 1
        except sqlite3.IntegrityError:
            skip += 1
    conn.commit()
    return ins, skip

print('DB and helpers ready')

In [ ]:
# Cell 3: Crawl RealSelf with stealth Playwright
# Expected: ~20-50 pairs per treatment URL, up to 10 pages each

from playwright.sync_api import sync_playwright

total_inserted = 0

with sync_playwright() as pw:
    browser = pw.chromium.launch(
        headless=True,
        args=[
            '--no-sandbox',
            '--disable-setuid-sandbox',
            '--disable-blink-features=AutomationControlled',
        ],
    )
    ctx = browser.new_context(
        user_agent=(
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/120.0.0.0 Safari/537.36'
        ),
        viewport={'width': 1920, 'height': 1080},
        locale='en-US',
        timezone_id='America/New_York',
        extra_http_headers={
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Sec-Fetch-Site': 'none',
            'Sec-Fetch-Mode': 'navigate',
        },
    )
    # Remove webdriver fingerprint
    ctx.add_init_script("""
        Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
        window.chrome = { runtime: {} };
        Object.defineProperty(navigator, 'plugins', { get: () => [1, 2, 3, 4] });
        Object.defineProperty(navigator, 'languages', { get: () => ['en-US', 'en'] });
    """)

    page = ctx.new_page()

    # Warm up: visit homepage first to get cookies
    page.goto('https://www.realself.com', wait_until='domcontentloaded', timeout=30000)
    time.sleep(random.uniform(3, 5))

    for base_url in SEED_URLS:
        treatment_slug = base_url.split('/')[-1]
        treatment_total = 0

        for pg in range(1, 11):   # up to 10 pages per treatment
            url = f'{base_url}?page={pg}'
            try:
                page.goto(url, wait_until='domcontentloaded', timeout=30000)
                time.sleep(random.uniform(2, 3))   # let lazy-load settle
                html = page.content()

                pairs = extract_pairs(html, url)
                ins, skip = save_pairs(pairs)
                total_inserted += ins
                treatment_total += ins

                print(f'  {treatment_slug} p{pg}: {ins} new | total={total_inserted}')

                if len(pairs) == 0:
                    break   # no results on this page — stop paginating

            except Exception as exc:
                print(f'  ERROR {url}: {exc}')
                break

            time.sleep(RATE_LIMIT + random.uniform(0, 3))

        print(f'>>> {treatment_slug}: {treatment_total} pairs')

    browser.close()

conn.close()
size_kb = DB_PATH.stat().st_size / 1024
print(f'\nCrawl complete — {total_inserted} pairs saved to {DB_PATH} ({size_kb:.0f} KB)')

In [ ]:
# Cell 4: Download the database
# After downloading, use scripts/merge_queue.py to merge with your local staging.db

from google.colab import files
files.download('/content/realself_pairs.db')
print('Downloaded realself_pairs.db')
print('Next: copy to data/realself_pairs.db, then run:')
print('  py -3.10 scripts/merge_queue.py --realself data/realself_pairs.db')